# Opus Lower — ConvNeXtV2-Base Inference & Evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import time
import numpy as np
from pathlib import Path

import torch
import torch.nn.functional as F
import timm
import onnx
import onnxruntime as ort

# ============================================================
# CONFIGURATION
# ============================================================
LOWER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")
MODELS = {
    "lower_v3": {
        "ckpt": LOWER_DIR / "best_lower_cnn.pth",
        "backbone": "convnextv2_base.fcmae_ft_in22k_in1k_384",
        "img_size": 384,
        "classes": ["jeans", "jogging_bottoms", "skirts", "trousers"],
        "onnx_out": LOWER_DIR / "lower_v3.onnx",
        "thresholds": LOWER_DIR / "lower_per_class_thresholds.json",
    },
}
OPSET = 17
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def load_pytorch_model(cfg):
    """Load a trained PyTorch model from checkpoint."""
    ckpt = torch.load(cfg["ckpt"], map_location=DEVICE, weights_only=False)

    model = timm.create_model(
        cfg["backbone"],
        pretrained=False,
        num_classes=len(cfg["classes"])
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(DEVICE)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Loaded: {cfg['backbone']} — {n_params:.1f}M params")
    print(f"  Best F1 (train): {ckpt.get('best_f1', 'N/A')}")
    return model

def export_to_onnx(model, cfg, opset=17):
    """Export PyTorch model to ONNX with dynamic batch size."""
    img_size = cfg["img_size"]
    onnx_path = cfg["onnx_out"]

    # Dummy input
    dummy = torch.randn(1, 3, img_size, img_size, device=DEVICE)

    # Export
    print(f"  Exporting to: {onnx_path}")
    torch.onnx.export(
        model,
        dummy,
        str(onnx_path),
        opset_version=opset,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"},
        },
    )

    # Validate
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)

    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"  ✓ ONNX valid — {size_mb:.1f} MB")
    print(f"  Opset: {opset}")
    print(f"  Input:  input  → [batch, 3, {img_size}, {img_size}]")
    print(f"  Output: logits → [batch, {len(cfg['classes'])}]")

    return onnx_path

def validate_onnx(model, cfg):
    """Compare PyTorch vs ONNX Runtime outputs."""
    img_size = cfg["img_size"]
    onnx_path = str(cfg["onnx_out"])

    # PyTorch inference
    dummy = torch.randn(1, 3, img_size, img_size, device=DEVICE)
    with torch.no_grad():
        pt_logits = model(dummy).cpu().numpy()

    # ONNX Runtime inference
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(onnx_path, providers=providers)
    ort_logits = session.run(
        ["logits"],
        {"input": dummy.cpu().numpy()}
    )[0]

    # Compare
    max_diff = np.abs(pt_logits - ort_logits).max()
    mean_diff = np.abs(pt_logits - ort_logits).mean()
    pt_pred = pt_logits.argmax(1)
    ort_pred = ort_logits.argmax(1)
    preds_match = (pt_pred == ort_pred).all()

    print(f"  Max  abs diff: {max_diff:.8f}")
    print(f"  Mean abs diff: {mean_diff:.8f}")
    print(f"  Predictions match: {preds_match}")

    if max_diff < 1e-4:
        print(f"  ✓ Outputs match within tolerance")
    else:
        print(f"  ⚠ Outputs differ beyond 1e-4 — check model")

    return session

def benchmark_onnx(cfg, n_warmup=10, n_runs=50):
    """Benchmark ONNX Runtime latency."""
    img_size = cfg["img_size"]
    onnx_path = str(cfg["onnx_out"])

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(onnx_path, providers=providers)

    active = session.get_providers()
    print(f"  Active providers: {active}")

    dummy_np = np.random.randn(1, 3, img_size, img_size).astype(np.float32)

    # Warmup
    for _ in range(n_warmup):
        session.run(["logits"], {"input": dummy_np})

    # Benchmark
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(["logits"], {"input": dummy_np})
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    print(f"  Latency (ms): mean={times.mean():.1f}, median={np.median(times):.1f}, "
          f"p95={np.percentile(times, 95):.1f}, min={times.min():.1f}, max={times.max():.1f}")
    print(f"  Throughput: ~{1000/times.mean():.0f} img/s (batch=1)")

    return times

# ============================================================
# MAIN EXECUTION
# ============================================================
print("=" * 60)
print("  LOWER V3 — ConvNeXt-V2-Base (384px)")
print("=" * 60)

cfg = MODELS["lower_v3"]
lower_model = load_pytorch_model(cfg)
print()
export_to_onnx(lower_model, cfg, opset=OPSET)
print()
validate_onnx(lower_model, cfg)
print()
benckmark_onnx(cfg)

# ============================================================
# METADATA GENERATION
# ============================================================
for name, cfg_meta in MODELS.items(): # Renamed cfg to cfg_meta to avoid conflict with main cfg
    # Load thresholds if available
    thresholds = {}
    if cfg_meta["thresholds"].exists():
        with open(cfg_meta["thresholds"]) as f:
            tdata = json.load(f)
        thresholds = tdata.get("thresholds", {})

    meta = {
        "model_name": name,
        "backbone": cfg_meta["backbone"],
        "onnx_file": cfg_meta["onnx_out"].name,
        "opset": OPSET,
        "input_name": "input",
        "output_name": "logits",
        "input_shape": [1, 3, cfg_meta["img_size"], cfg_meta["img_size"]],
        "dynamic_batch": True,
        "classes": cfg_meta["classes"],
        "num_classes": len(cfg_meta["classes"]),
        "preprocessing": {
            "resize": [cfg_meta["img_size"], cfg_meta["img_size"]],
            "normalize_mean": [0.485, 0.456, 0.406],
            "normalize_std": [0.229, 0.224, 0.225],
            "channel_order": "RGB",
            "tensor_format": "NCHW",
            "dtype": "float32",
        },
        "thresholds": thresholds,
    }

    meta_path = cfg_meta["onnx_out"].with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"✓ {name}: {meta_path}")
    print(f"  Classes: {cfg_meta['classes']}")
    print(f"  Thresholds: {thresholds}")
    print()

print("\n" + "=" * 60)
print("  EXPORT SUMMARY")
print("=" * 60)

for name, cfg_summary in MODELS.items(): # Renamed cfg to cfg_summary
    onnx_path = cfg_summary["onnx_out"]
    meta_path = onnx_path.with_suffix(".json")
    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)

    print(f"\n  {name}:")
    print(f"    ONNX:      {onnx_path} ({size_mb:.1f} MB)")
    print(f"    Metadata:  {meta_path}")
    print(f"    Backbone:  {cfg_summary['backbone']}")
    print(f"    Input:     [batch, 3, {cfg_summary['img_size']}, {cfg_summary['img_size']}]")
    print(f"    Classes:   {cfg_summary['classes']}")

print(f"\n{'='*60}")
print("  All exports complete!")
print("="*60)

In [ ]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ============================================================
# CONFIG
# ============================================================
# --- TEST SETS ---
TEST_ROOT_1 = Path("/content/drive/Shareddrives/Garment Type/classified_images_13_14_test/13_14th_test")
TEST_ROOT_2 = Path("/content/drive/Shareddrives/Garment Type/Careys_labelled_data/to_test")

# --- MODEL ---
MODEL_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")
CKPT_PATH = MODEL_DIR / "best_lower_cnn.pth"
THRESH_PATH = MODEL_DIR / "lower_per_class_thresholds.json"

OUTDIR = MODEL_DIR / "inference_results"
OUTDIR.mkdir(exist_ok=True, parents=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# These must match training
IMG_SIZE = 384
BACKBONE = "convnextv2_base.fcmae_ft_in22k_in1k_384"

print(f"Device: {DEVICE}")

In [ ]:
# ============================================================
# PREPROCESS
# ============================================================
def build_infer_aug(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ToTensorV2()
    ])


# ============================================================
# LOAD MODEL
# ============================================================
def load_model():
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]

    model = timm.create_model(
        BACKBONE,
        pretrained=False,
        num_classes=len(classes)
    ).to(DEVICE)

    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()

    print(f"Loaded: {CKPT_PATH.name}")
    print(f"Backbone: {BACKBONE}, Resolution: {IMG_SIZE}")
    print(f"Classes: {classes}")
    print(f"Best F1 (train): {ckpt.get('best_f1', 'N/A')}")

    return model, classes


# ============================================================
# LOAD THRESHOLDS
# ============================================================
def load_thresholds():
    if THRESH_PATH.exists():
        with open(THRESH_PATH) as f:
            data = json.load(f)
        thresholds = data["thresholds"]
        print(f"\nPer-class thresholds loaded:")
        for cls, t in thresholds.items():
            print(f"  {cls:<18s}: {t:.2f}")
        return thresholds
    else:
        print(f"\nWARNING: No threshold file found at {THRESH_PATH}")
        print("Running without rejection.")
        return None

In [ ]:
# ============================================================
# COLLECT TEST SAMPLES
# ============================================================
def collect_test_samples(test_root):
    samples = []
    lower_dir = test_root / "lower"
    if not lower_dir.exists():
        print(f"WARNING: {lower_dir} not found, skipping.")
        return samples

    for cls in sorted(os.listdir(lower_dir)):
        cls_dir = lower_dir / cls
        if not cls_dir.is_dir():
            continue
        for img in cls_dir.iterdir():
            if img.is_file():
                samples.append((str(img), cls.lower()))

    return samples


# ============================================================
# INFERENCE
# ============================================================
def infer_on_samples(samples, model, classes, aug):
    cls2idx = {c.lower(): i for i, c in enumerate(classes)}
    y_true, y_pred = [], []
    all_probs = []

    with torch.no_grad():
        for path, gt in tqdm(samples):
            try:
                img_pil = Image.open(path).convert("RGB")
                img_np = np.array(img_pil)
            except:
                print(f"  Skipping corrupt image: {path}")
                continue

            x = aug(image=img_np)["image"].unsqueeze(0).to(DEVICE)

            with torch.cuda.amp.autocast():
                logits = model(x)[0]

            probs = torch.softmax(logits, dim=0).cpu().numpy()
            pred = probs.argmax()

            if gt.lower() not in cls2idx:
                print(f"  Skipping unknown GT class: {gt}")
                continue

            y_true.append(cls2idx[gt.lower()])
            y_pred.append(pred)
            all_probs.append(probs)

    return np.array(y_true), np.array(y_pred), np.array(all_probs)

In [ ]:
# ============================================================
# APPLY THRESHOLDS
# ============================================================
def apply_thresholds(y_true, y_pred, probs, classes, thresholds):
    final_preds = []
    decisions = []

    for i in range(len(y_pred)):
        top_idx = y_pred[i]
        top_conf = probs[i, top_idx]
        top_class = classes[top_idx]

        if top_conf >= thresholds.get(top_class, 0.5):
            final_preds.append(top_idx)
            decisions.append("confident")
        else:
            final_preds.append(-1)
            decisions.append("rejected")

    return np.array(final_preds), np.array(decisions)


# ============================================================
# PLOTTING
# ============================================================
def plot_confusion_matrix(cm, classes, title, save_path=None):
    plt.figure(figsize=(7, 5.5))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha="right")
    plt.yticks(tick_marks, classes)

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i, j]),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

In [ ]:
# ============================================================
# REPORT WITH REJECTION
# ============================================================
def report_with_rejection(y_true, final_preds, decisions, classes, title=""):
    n_total = len(y_true)
    n_rejected = (final_preds == -1).sum()
    n_accepted = n_total - n_rejected

    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  Total:    {n_total}")
    print(f"  Accepted: {n_accepted}  ({n_accepted/n_total*100:.1f}%)")
    print(f"  Rejected: {n_rejected}  ({n_rejected/n_total*100:.1f}%)")

    for d in ["confident", "rejected"]:
        count = (decisions == d).sum()
        print(f"    {d:>12s}: {count:5d}  ({count/n_total*100:.1f}%)")

    accepted_mask = final_preds != -1
    report = ""

    if accepted_mask.sum() > 0:
        acc_true = y_true[accepted_mask]
        acc_pred = final_preds[accepted_mask]
        print(f"\n--- Classification Report (accepted only) ---")
        report = classification_report(
            acc_true, acc_pred, target_names=classes, digits=4, zero_division=0
        )
        print(report)

        acc = (acc_true == acc_pred).sum() / len(acc_true)
        print(f"  Accuracy (accepted): {acc:.4f}")

    all_correct = ((final_preds == y_true) & (final_preds != -1)).sum()
    print(f"  Accuracy (all, rejected=wrong): {all_correct/n_total:.4f}")

    return report

In [ ]:
# ============================================================
# RUN ON A SINGLE TEST SET
# ============================================================
def evaluate_test_set(name, test_root, model, classes, aug, thresholds):
    print(f"\n{'#'*60}")
    print(f"  TEST SET: {name}")
    print(f"  Path: {test_root}")
    print(f"{'#'*60}")

    samples = collect_test_samples(test_root)
    if len(samples) == 0:
        print("  No samples found, skipping.")
        return

    print(f"  Found {len(samples)} samples")

    # Print class distribution
    from collections import Counter
    dist = Counter(c for _, c in samples)
    for cls, cnt in sorted(dist.items()):
        print(f"    {cls:18s}: {cnt}")

    y_true, y_pred, probs = infer_on_samples(samples, model, classes, aug)

    # --- Baseline (no thresholds) ---
    print(f"\n--- BASELINE (no rejection) ---")
    report_base = classification_report(y_true, y_pred, target_names=classes, digits=4)
    print(report_base)

    cm_base = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    plot_confusion_matrix(
        cm_base, classes,
        f"{name} — Baseline (no rejection)",
        save_path=OUTDIR / f"{name}_baseline_cm.png"
    )

    (OUTDIR / f"{name}_baseline_report.txt").write_text(report_base)

    # --- With thresholds ---
    if thresholds:
        final_preds, decisions = apply_thresholds(y_true, y_pred, probs, classes, thresholds)

        report_thresh = report_with_rejection(
            y_true, final_preds, decisions, classes,
            title=f"{name} — With Per-Class Thresholds"
        )

        accepted_mask = final_preds != -1
        if accepted_mask.sum() > 0:
            cm_thresh = confusion_matrix(
                y_true[accepted_mask], final_preds[accepted_mask],
                labels=list(range(len(classes)))
            )
            plot_confusion_matrix(
                cm_thresh, classes,
                f"{name} — With Thresholds (accepted only)",
                save_path=OUTDIR / f"{name}_thresholded_cm.png"
            )

        (OUTDIR / f"{name}_thresholded_report.txt").write_text(str(report_thresh))

        # Save per-sample decisions
        decisions_path = OUTDIR / f"{name}_sample_decisions.csv"
        with open(decisions_path, "w") as f:
            f.write("sample_idx,true_class,raw_pred,final_pred,decision,confidence\n")
            for i in range(len(y_true)):
                true_cls = classes[y_true[i]]
                raw_cls = classes[y_pred[i]]
                final_cls = classes[final_preds[i]] if final_preds[i] != -1 else "REJECTED"
                conf = probs[i, y_pred[i]]
                f.write(f"{i},{true_cls},{raw_cls},{final_cls},{decisions[i]},{conf:.4f}\n")

        print(f"  Per-sample decisions: {decisions_path}")

    return y_true, y_pred, probs

In [ ]:
# ============================================================
# MAIN
# ============================================================
model, classes = load_model()
aug = build_infer_aug(IMG_SIZE)
thresholds = load_thresholds()

# Run on both test sets
r1 = evaluate_test_set("test1_13_14th", TEST_ROOT_1, model, classes, aug, thresholds)
r2 = evaluate_test_set("test2_careys", TEST_ROOT_2, model, classes, aug, thresholds)

# Combined report
if r1 and r2:
    y1_true, y1_pred, p1 = r1
    y2_true, y2_pred, p2 = r2

    y_all_true = np.concatenate([y1_true, y2_true])
    y_all_pred = np.concatenate([y1_pred, y2_pred])
    p_all = np.concatenate([p1, p2])

    print(f"\n{'#'*60}")
    print(f"  COMBINED (both test sets)")
    print(f"{'#'*60}")

    report_combined = classification_report(
        y_all_true, y_all_pred, target_names=classes, digits=4
    )
    print(f"\n--- Baseline Combined ---")
    print(report_combined)

    cm_combined = confusion_matrix(y_all_true, y_all_pred, labels=list(range(len(classes))))
    plot_confusion_matrix(
        cm_combined, classes,
        "Combined — Baseline",
        save_path=OUTDIR / "combined_baseline_cm.png"
    )

    (OUTDIR / "combined_baseline_report.txt").write_text(report_combined)

    if thresholds:
        final_all, dec_all = apply_thresholds(y_all_true, y_all_pred, p_all, classes, thresholds)
        report_with_rejection(
            y_all_true, final_all, dec_all, classes,
            title="Combined — With Per-Class Thresholds"
        )

print(f"\nAll outputs saved in: {OUTDIR}")

In [ ]:
import os
import json
import timeimport os
import shutil
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import timm
import onnx
from onnxsim import simplify
import onnxruntime as ort

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPSET = 17

# >>>>>>>>>> EDIT PATHS HERE <<<<<<<<
LOWER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")
CKPT      = LOWER_DIR / "best_lower_cnn.pth"
BACKBONE  = "convnextv2_base.fcmae_ft_in22k_in1k_384"
IMG_SIZE  = 384
CLASSES   = ["jeans", "jogging_bottoms", "skirts", "trousers"]
ONNX_NAME = "lower_v3_trt.onnx"


class TRTCompatibleWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        x = self.model.forward_features(x)
        x = x.mean(dim=(-2, -1), keepdim=True)
        head = self.model.head
        if hasattr(head, 'norm') and head.norm is not None:
            x = head.norm(x)
        x = x.flatten(1)
        if hasattr(head, 'pre_logits') and head.pre_logits is not None:
            x = head.pre_logits(x)
        if hasattr(head, 'drop') and head.drop is not None:
            x = head.drop(x)
        if hasattr(head, 'fc') and head.fc is not None:
            x = head.fc(x)
        return x


# Load model
ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model = timm.create_model(BACKBONE, pretrained=False, num_classes=len(CLASSES))
model.load_state_dict(ckpt["model_state"], strict=True)
model.to(DEVICE).eval()
print(f"✓ Loaded (F1={ckpt.get('best_f1', 'N/A')})")

# Validate wrapper
wrapper = TRTCompatibleWrapper(model).to(DEVICE).eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    diff = (model(dummy) - wrapper(dummy)).abs().max().item()
print(f"Wrapper diff: {diff:.10f}")
assert diff < 1e-4, f"Mismatch: {diff}"
print("✓ Wrapper validated")

# Export to local
local_onnx = f"/content/{ONNX_NAME}"
torch.onnx.export(
    wrapper, dummy, local_onnx,
    opset_version=OPSET,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    do_constant_folding=True, dynamo=False,
)

# Simplify
onnx_model = onnx.load(local_onnx)
onnx_sim, check = simplify(onnx_model)
assert check, "onnxsim failed"
onnx.save(onnx_sim, local_onnx)

# Check ops
ops = set(n.op_type for n in onnx.load(local_onnx).graph.node)
bad = ops & {"SequenceEmpty", "SequenceConstruct", "SequenceAt", "SequenceInsert", "ConcatFromSequence"}
print(f"Ops: {sorted(ops)}")
print(f"✓ No bad ops" if not bad else f"⚠ Bad ops: {bad}")
print(f"Local: {local_onnx} ({os.path.getsize(local_onnx)/(1024*1024):.1f} MB)")

# Validate ONNX vs PyTorch
session = ort.InferenceSession(local_onnx, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
with torch.no_grad():
    pt = model(dummy).cpu().numpy()
ort_out = session.run(["logits"], {"input": dummy.cpu().numpy()})[0]
print(f"Max diff: {np.abs(pt - ort_out).max():.8f} | Preds match: {(pt.argmax(1) == ort_out.argmax(1)).all()}")

# Dynamic batch test
batch_out = session.run(["logits"], {"input": torch.randn(4, 3, IMG_SIZE, IMG_SIZE).numpy()})[0]
print(f"✓ Dynamic batch: {batch_out.shape}")
del session

# Copy to Drive
final = str(LOWER_DIR / ONNX_NAME)
for f in [final, final + ".data"]:
    if os.path.exists(f):
        os.remove(f)
shutil.copy2(local_onnx, final)
if os.path.exists(local_onnx + ".data"):
    shutil.copy2(local_onnx + ".data", final + ".data")
    print(f"✓ Copied to Drive: {final} + .data")
else:
    print(f"✓ Copied to Drive: {final} (single file)")

print(f"\n✅ READY — copy to Jetson and run:")
print(f"  /usr/src/tensorrt/bin/trtexec \\")
print(f"      --onnx={ONNX_NAME} --saveEngine=lower_v3.engine \\")
print(f"      --fp16 --minShapes=input:1x3x384x384 \\")
print(f"      --optShapes=input:1x3x384x384 --maxShapes=input:4x3x384x384 \\")
print(f"      --memPoolSize=workspace:2048M")
from pathlib import Path

import numpy as np
import torch
import timm
!pip install onnx onnxruntime onnxscript
import matplotlib.pyplot as plt
import onnx
import onnxruntime as ort

# ============================================================
# CONFIGURATION
# ============================================================
LOWER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")

MODELS = {
    "lower_v3": {
        "ckpt": LOWER_DIR / "best_lower_cnn.pth",
        "backbone": "convnextv2_base.fcmae_ft_in22k_in1k_384",
        "img_size": 384,
        "classes": ["jeans", "jogging_bottoms", "skirts", "trousers"],
        "onnx_out": LOWER_DIR / "lower_v3.onnx",
        "thresholds": LOWER_DIR / "lower_per_class_thresholds.json",
    },
}

OPSET = 17
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXPORT_BATCH = 4
VERIFY_BATCHES = [1, 4]


# ============================================================
# HELPER FUNCTIONS
# ============================================================
def load_pytorch_model(cfg):
    """Load a trained PyTorch model from checkpoint."""
    ckpt = torch.load(cfg["ckpt"], map_location=DEVICE, weights_only=False)

    model = timm.create_model(
        cfg["backbone"],
        pretrained=False,
        num_classes=len(cfg["classes"])
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(DEVICE)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Loaded: {cfg['backbone']} — {n_params:.1f}M params")
    print(f"  Best F1 (train): {ckpt.get('best_f1', 'N/A')}")
    return model, ckpt


def inspect_onnx_io(onnx_path):
    """Print ONNX input/output shapes."""
    model = onnx.load(str(onnx_path))

    def get_shape(value_info):
        dims = []
        for d in value_info.type.tensor_type.shape.dim:
            if d.dim_param:
                dims.append(d.dim_param)
            elif d.dim_value:
                dims.append(d.dim_value)
            else:
                dims.append("?")
        return dims

    print("\n  ONNX I/O:")
    for x in model.graph.input:
        print(f"    Input  {x.name}: {get_shape(x)}")
    for x in model.graph.output:
        print(f"    Output {x.name}: {get_shape(x)}")


def export_to_onnx(model, cfg, opset=17):
    """Export PyTorch model to ONNX with dynamic batch size."""
    img_size = cfg["img_size"]
    onnx_path = cfg["onnx_out"]

    # Use batch > 1 to help expose static-batch problems early
    dummy = torch.randn(EXPORT_BATCH, 3, img_size, img_size, device=DEVICE)

    print(f"  Exporting to: {onnx_path}")
    torch.onnx.export(
        model,
        dummy,
        str(onnx_path),
        opset_version=opset,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"},
        },
        do_constant_folding=True,
    )

    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)

    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"  ✓ ONNX valid — {size_mb:.1f} MB")
    print(f"  Opset: {opset}")
    print(f"  Input:  input  → [batch, 3, {img_size}, {img_size}]")
    print(f"  Output: logits → [batch, {len(cfg['classes'])}]")

    inspect_onnx_io(onnx_path)
    return onnx_path


def simplify_onnx(cfg):
    """Simplify ONNX graph to reduce TRT foreign-node issues."""
    onnx_path = cfg["onnx_out"]
    sim_path = onnx_path.with_name(onnx_path.stem + "_sim.onnx")

    try:
        import onnxsim
        print(f"  Simplifying ONNX → {sim_path}")
        model = onnx.load(str(onnx_path))
        sim_model, check = onnxsim.simplify(model)

        if not check:
            print("  ⚠ onnxsim reported failed validation; keeping original ONNX")
            return onnx_path

        onnx.save(sim_model, str(sim_path))
        size_mb = os.path.getsize(sim_path) / (1024 * 1024)
        print(f"  ✓ Simplified ONNX saved — {size_mb:.1f} MB")
        return sim_path

    except ImportError:
        print("  ⚠ onnxsim not installed; skipping simplification")
        return onnx_path
    except Exception as e:
        print(f"  ⚠ ONNX simplification failed: {e}")
        return onnx_path


def validate_onnx(model, cfg, onnx_path):
    """Compare PyTorch vs ONNX Runtime outputs for multiple batch sizes."""
    img_size = cfg["img_size"]

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(str(onnx_path), providers=providers)

    print(f"  Active providers: {session.get_providers()}")

    for batch_size in VERIFY_BATCHES:
        dummy = torch.randn(batch_size, 3, img_size, img_size, device=DEVICE)

        with torch.no_grad():
            pt_logits = model(dummy).detach().cpu().numpy()

        ort_logits = session.run(
            ["logits"],
            {"input": dummy.detach().cpu().numpy()}
        )[0]

        max_diff = np.abs(pt_logits - ort_logits).max()
        mean_diff = np.abs(pt_logits - ort_logits).mean()
        pt_pred = pt_logits.argmax(1)
        ort_pred = ort_logits.argmax(1)
        preds_match = bool((pt_pred == ort_pred).all())

        print(f"\n  Batch {batch_size} validation:")
        print(f"    PyTorch shape: {pt_logits.shape}")
        print(f"    ONNX shape:    {ort_logits.shape}")
        print(f"    Max  abs diff: {max_diff:.8f}")
        print(f"    Mean abs diff: {mean_diff:.8f}")
        print(f"    Predictions match: {preds_match}")

        if max_diff < 1e-4:
            print("    ✓ Outputs match within tolerance")
        else:
            print("    ⚠ Outputs differ beyond 1e-4 — inspect graph/export")

    return session


def benchmark_onnx(cfg, onnx_path, batch_size=1, n_warmup=10, n_runs=50):
    """Benchmark ONNX Runtime latency."""
    img_size = cfg["img_size"]

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(str(onnx_path), providers=providers)

    active = session.get_providers()
    print(f"  Active providers: {active}")

    dummy_np = np.random.randn(batch_size, 3, img_size, img_size).astype(np.float32)

    for _ in range(n_warmup):
        session.run(["logits"], {"input": dummy_np})

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(["logits"], {"input": dummy_np})
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    print(
        f"  Latency (ms): mean={times.mean():.1f}, median={np.median(times):.1f}, "
        f"p95={np.percentile(times, 95):.1f}, min={times.min():.1f}, max={times.max():.1f}"
    )
    print(f"  Throughput: ~{1000 / times.mean() * batch_size:.0f} img/s (batch={batch_size})")

    return times


def make_trt_commands(cfg, onnx_path):
    img_size = cfg["img_size"]
    stem = onnx_path.stem

    fp16 = (
        f"trtexec --onnx={onnx_path.name} "
        f"--saveEngine={stem}_fp16.engine "
        f"--minShapes=input:1x3x{img_size}x{img_size} "
        f"--optShapes=input:4x3x{img_size}x{img_size} "
        f"--maxShapes=input:4x3x{img_size}x{img_size} "
        f"--fp16 --verbose"
    )
    fp32 = (
        f"trtexec --onnx={onnx_path.name} "
        f"--saveEngine={stem}_fp32.engine "
        f"--minShapes=input:1x3x{img_size}x{img_size} "
        f"--optShapes=input:4x3x{img_size}x{img_size} "
        f"--maxShapes=input:4x3x{img_size}x{img_size} "
        f"--verbose"
    )
    return fp16, fp32


# ============================================================
# MAIN EXECUTION
# ============================================================
print("=" * 60)
print("  LOWER V3 — ConvNeXt-V2-Base (384px)")
print("=" * 60)

cfg = MODELS["lower_v3"]
lower_model, ckpt = load_pytorch_model(cfg)
print()

onnx_path = export_to_onnx(lower_model, cfg, opset=OPSET)
print()

onnx_path_for_trt = simplify_onnx(cfg)
print()

validate_onnx(lower_model, cfg, onnx_path_for_trt)
print()

benchmark_onnx(cfg, onnx_path_for_trt, batch_size=1)
print()
benchmark_onnx(cfg, onnx_path_for_trt, batch_size=4)

# ============================================================
# METADATA GENERATION
# ============================================================
for name, cfg_meta in MODELS.items():
    thresholds = {}
    if cfg_meta["thresholds"].exists():
        with open(cfg_meta["thresholds"]) as f:
            tdata = json.load(f)
        thresholds = tdata.get("thresholds", {})

    final_onnx = cfg_meta["onnx_out"].with_name(cfg_meta["onnx_out"].stem + "_sim.onnx")
    if not final_onnx.exists():
        final_onnx = cfg_meta["onnx_out"]

    fp16_cmd, fp32_cmd = make_trt_commands(cfg_meta, final_onnx)

    meta = {
        "model_name": name,
        "backbone": cfg_meta["backbone"],
        "onnx_file": final_onnx.name,
        "opset": OPSET,
        "input_name": "input",
        "output_name": "logits",
        "input_shape": ["N", 3, cfg_meta["img_size"], cfg_meta["img_size"]],
        "output_shape": ["N", len(cfg_meta["classes"])],
        "dynamic_batch": True,
        "classes": cfg_meta["classes"],
        "num_classes": len(cfg_meta["classes"]),
        "preprocessing": {
            "resize": [cfg_meta["img_size"], cfg_meta["img_size"]],
            "normalize_mean": [0.485, 0.456, 0.406],
            "normalize_std": [0.229, 0.224, 0.225],
            "channel_order": "RGB",
            "tensor_format": "NCHW",
            "dtype": "float32",
        },
        "thresholds": thresholds,
        "training_info": {
            "best_f1": ckpt.get("best_f1"),
        },
        "trt_conversion": {
            "fp16_command": fp16_cmd,
            "fp32_command": fp32_cmd,
            "note": "Use explicit dynamic shape profiles for batch 1..4.",
        },
    }

    meta_path = cfg_meta["onnx_out"].with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"✓ {name}: {meta_path}")
    print(f"  Classes: {cfg_meta['classes']}")
    print(f"  Thresholds: {thresholds}")
    print(f"  TRT FP16: {fp16_cmd}")
    print(f"  TRT FP32: {fp32_cmd}")
    print()

print("\n" + "=" * 60)
print("  EXPORT SUMMARY")
print("=" * 60)

for name, cfg_summary in MODELS.items():
    final_onnx = cfg_summary["onnx_out"].with_name(cfg_summary["onnx_out"].stem + "_sim.onnx")
    if not final_onnx.exists():
        final_onnx = cfg_summary["onnx_out"]

    meta_path = cfg_summary["onnx_out"].with_suffix(".json")
    size_mb = os.path.getsize(final_onnx) / (1024 * 1024)

    print(f"\n  {name}:")
    print(f"    ONNX:      {final_onnx} ({size_mb:.1f} MB)")
    print(f"    Metadata:  {meta_path}")
    print(f"    Backbone:  {cfg_summary['backbone']}")
    print(f"    Input:     [batch, 3, {cfg_summary['img_size']}, {cfg_summary['img_size']}]")
    print(f"    Classes:   {cfg_summary['classes']}")

print(f"\n{'=' * 60}")
print("  All exports complete!")
print("=" * 60)

In [ ]:
import os
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import timm

# Notebook installs
# !pip install -q onnx onnxruntime onnxsim

import onnx
import onnxruntime as ort

# ============================================================
# CONFIGURATION
# ============================================================
LOWER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")

MODELS = {
    "lower_v3": {
        "ckpt": LOWER_DIR / "best_lower_cnn.pth",
        "backbone": "convnextv2_base.fcmae_ft_in22k_in1k_384",
        "img_size": 384,
        "classes": ["jeans", "jogging_bottoms", "skirts", "trousers"],
        "onnx_out": LOWER_DIR / "lower_v3_trt.onnx",
        "thresholds": LOWER_DIR / "lower_per_class_thresholds.json",
    },
}

# IMPORTANT: use lower opset for TRT compatibility on this CNN
OPSET = 13
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXPORT_BATCH = 4
VERIFY_BATCHES = [1, 4]


# ============================================================
# MODEL WRAPPER
# ============================================================
class TRTExportWrapper(nn.Module):
    """Simple tensor-only forward path for tracing/export."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)


# ============================================================
# HELPERS
# ============================================================
def load_pytorch_model(cfg):
    ckpt = torch.load(cfg["ckpt"], map_location=DEVICE, weights_only=False)

    model = timm.create_model(
        cfg["backbone"],
        pretrained=False,
        num_classes=len(cfg["classes"])
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(DEVICE)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Loaded: {cfg['backbone']} — {n_params:.1f}M params")
    print(f"  Best F1 (train): {ckpt.get('best_f1', 'N/A')}")
    return model, ckpt


def get_shape(value_info):
    dims = []
    for d in value_info.type.tensor_type.shape.dim:
        if d.dim_param:
            dims.append(d.dim_param)
        elif d.dim_value:
            dims.append(d.dim_value)
        else:
            dims.append("?")
    return dims


def inspect_onnx_io(onnx_path):
    model = onnx.load(str(onnx_path))
    print("\n  ONNX I/O:")
    for x in model.graph.input:
        print(f"    Input  {x.name}: {get_shape(x)}")
    for x in model.graph.output:
        print(f"    Output {x.name}: {get_shape(x)}")


def print_onnx_op_histogram(onnx_path):
    model = onnx.load(str(onnx_path))
    ops = {}
    for node in model.graph.node:
        ops[node.op_type] = ops.get(node.op_type, 0) + 1

    print("\n  ONNX op histogram:")
    for op in sorted(ops):
        print(f"    {op}: {ops[op]}")


def find_bad_ops(onnx_path, bad_ops=("Loop", "SequenceEmpty", "If")):
    model = onnx.load(str(onnx_path))
    found = sorted(set(node.op_type for node in model.graph.node if node.op_type in bad_ops))
    return found


def export_to_onnx(model, cfg, opset=13):
    img_size = cfg["img_size"]
    onnx_path = cfg["onnx_out"]

    wrapper = TRTExportWrapper(model).to(DEVICE).eval()

    # Export with batch > 1
    dummy = torch.randn(EXPORT_BATCH, 3, img_size, img_size, device=DEVICE)

    # Trace first to freeze control flow
    print("  Tracing model for export...")
    traced = torch.jit.trace(wrapper, dummy, strict=False).eval()

    print(f"  Exporting traced model to: {onnx_path}")
    torch.onnx.export(
        traced,
        dummy,
        str(onnx_path),
        opset_version=opset,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"},
        },
        do_constant_folding=True,
        training=torch.onnx.TrainingMode.EVAL,
        keep_initializers_as_inputs=False,
        dynamo=False,
    )

    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)

    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"  ✓ ONNX valid — {size_mb:.1f} MB")
    print(f"  Opset: {opset}")
    print(f"  Input:  input  → [batch, 3, {img_size}, {img_size}]")
    print(f"  Output: logits → [batch, {len(cfg['classes'])}]")

    inspect_onnx_io(onnx_path)
    print_onnx_op_histogram(onnx_path)

    bad = find_bad_ops(onnx_path)
    print(f"  Problematic ops before simplification: {bad if bad else 'None'}")

    return onnx_path


def simplify_onnx(onnx_path):
    sim_path = onnx_path.with_name(onnx_path.stem + "_sim.onnx")

    try:
        import onnxsim
        print(f"\n  Simplifying ONNX → {sim_path}")
        model = onnx.load(str(onnx_path))
        sim_model, check = onnxsim.simplify(model)

        if not check:
            print("  ⚠ onnxsim validation failed; keeping original ONNX")
            return onnx_path

        onnx.save(sim_model, str(sim_path))
        size_mb = os.path.getsize(sim_path) / (1024 * 1024)
        print(f"  ✓ Simplified ONNX saved — {size_mb:.1f} MB")

        print_onnx_op_histogram(sim_path)
        bad = find_bad_ops(sim_path)
        print(f"  Problematic ops after simplification: {bad if bad else 'None'}")

        return sim_path

    except ImportError:
        print("  ⚠ onnxsim not installed; skipping simplification")
        return onnx_path
    except Exception as e:
        print(f"  ⚠ ONNX simplification failed: {e}")
        return onnx_path


def validate_onnx(model, cfg, onnx_path):
    img_size = cfg["img_size"]

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(str(onnx_path), providers=providers)

    print(f"  Active providers: {session.get_providers()}")

    for batch_size in VERIFY_BATCHES:
        dummy = torch.randn(batch_size, 3, img_size, img_size, device=DEVICE)

        with torch.no_grad():
            pt_logits = model(dummy).detach().cpu().numpy()

        ort_logits = session.run(
            ["logits"],
            {"input": dummy.detach().cpu().numpy()}
        )[0]

        max_diff = np.abs(pt_logits - ort_logits).max()
        mean_diff = np.abs(pt_logits - ort_logits).mean()
        pt_pred = pt_logits.argmax(1)
        ort_pred = ort_logits.argmax(1)
        preds_match = bool((pt_pred == ort_pred).all())

        print(f"\n  Batch {batch_size} validation:")
        print(f"    PyTorch shape: {pt_logits.shape}")
        print(f"    ONNX shape:    {ort_logits.shape}")
        print(f"    Max  abs diff: {max_diff:.8f}")
        print(f"    Mean abs diff: {mean_diff:.8f}")
        print(f"    Predictions match: {preds_match}")

        if max_diff < 1e-4:
            print("    ✓ Outputs match within tolerance")
        else:
            print("    ⚠ Outputs differ beyond 1e-4 — inspect export")

    return session


def benchmark_onnx(cfg, onnx_path, batch_size=1, n_warmup=10, n_runs=50):
    img_size = cfg["img_size"]

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(str(onnx_path), providers=providers)

    print(f"  Active providers: {session.get_providers()}")

    dummy_np = np.random.randn(batch_size, 3, img_size, img_size).astype(np.float32)

    for _ in range(n_warmup):
        session.run(["logits"], {"input": dummy_np})

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(["logits"], {"input": dummy_np})
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    print(
        f"  Latency (ms): mean={times.mean():.1f}, median={np.median(times):.1f}, "
        f"p95={np.percentile(times, 95):.1f}, min={times.min():.1f}, max={times.max():.1f}"
    )
    print(f"  Throughput: ~{1000 / times.mean() * batch_size:.0f} img/s (batch={batch_size})")
    return times


def make_trt_commands(cfg, onnx_path):
    img_size = cfg["img_size"]
    stem = onnx_path.stem

    fp16 = (
        f"trtexec --onnx={onnx_path.name} "
        f"--saveEngine={stem}_fp16.engine "
        f"--minShapes=input:1x3x{img_size}x{img_size} "
        f"--optShapes=input:4x3x{img_size}x{img_size} "
        f"--maxShapes=input:4x3x{img_size}x{img_size} "
        f"--fp16 --verbose"
    )
    fp32 = (
        f"trtexec --onnx={onnx_path.name} "
        f"--saveEngine={stem}_fp32.engine "
        f"--minShapes=input:1x3x{img_size}x{img_size} "
        f"--optShapes=input:4x3x{img_size}x{img_size} "
        f"--maxShapes=input:4x3x{img_size}x{img_size} "
        f"--verbose"
    )
    return fp16, fp32


# ============================================================
# MAIN EXECUTION
# ============================================================
print("=" * 60)
print("  LOWER V3 — ConvNeXt-V2-Base (384px) TRT-safe export")
print("=" * 60)

cfg = MODELS["lower_v3"]
lower_model, ckpt = load_pytorch_model(cfg)
print()

onnx_path = export_to_onnx(lower_model, cfg, opset=OPSET)
print()

onnx_path_for_trt = simplify_onnx(onnx_path)
print()

bad_ops = find_bad_ops(onnx_path_for_trt)
print(f"  Final problematic ops: {bad_ops if bad_ops else 'None'}")
if bad_ops:
    raise RuntimeError(
        f"Export still contains TRT-unfriendly ops: {bad_ops}. "
        f"Try a different backbone/export path if TRT build still fails."
    )

validate_onnx(lower_model, cfg, onnx_path_for_trt)
print()

benchmark_onnx(cfg, onnx_path_for_trt, batch_size=1)
print()
benchmark_onnx(cfg, onnx_path_for_trt, batch_size=4)

# ============================================================
# METADATA GENERATION
# ============================================================
thresholds = {}
if cfg["thresholds"].exists():
    with open(cfg["thresholds"]) as f:
        tdata = json.load(f)
    thresholds = tdata.get("thresholds", {})

fp16_cmd, fp32_cmd = make_trt_commands(cfg, onnx_path_for_trt)

meta = {
    "model_name": "lower_v3",
    "backbone": cfg["backbone"],
    "onnx_file": onnx_path_for_trt.name,
    "opset": OPSET,
    "input_name": "input",
    "output_name": "logits",
    "input_shape": ["N", 3, cfg["img_size"], cfg["img_size"]],
    "output_shape": ["N", len(cfg["classes"])],
    "dynamic_batch": True,
    "classes": cfg["classes"],
    "num_classes": len(cfg["classes"]),
    "preprocessing": {
        "resize": [cfg["img_size"], cfg["img_size"]],
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225],
        "channel_order": "RGB",
        "tensor_format": "NCHW",
        "dtype": "float32",
    },
    "thresholds": thresholds,
    "training_info": {
        "best_f1": ckpt.get("best_f1"),
    },
    "trt_conversion": {
        "fp16_command": fp16_cmd,
        "fp32_command": fp32_cmd,
        "note": "Use traced export + opset 13 + explicit TRT shape profiles.",
    },
}

meta_path = cfg["onnx_out"].with_suffix(".json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\n" + "=" * 60)
print("  EXPORT SUMMARY")
print("=" * 60)
size_mb = os.path.getsize(onnx_path_for_trt) / (1024 * 1024)

print(f"\n  lower_v3:")
print(f"    ONNX:      {onnx_path_for_trt} ({size_mb:.1f} MB)")
print(f"    Metadata:  {meta_path}")
print(f"    Backbone:  {cfg['backbone']}")
print(f"    Input:     [batch, 3, {cfg['img_size']}, {cfg['img_size']}]")
print(f"    Classes:   {cfg['classes']}")
print(f"    TRT FP16:  {fp16_cmd}")
print(f"    TRT FP32:  {fp32_cmd}")

print(f"\n{'=' * 60}")
print("  Export complete")
print("=" * 60)